In [1]:
from src.config import Config
import pandas as pd

csv_path = Config.raw_posts_file.value
df = pd.read_csv(csv_path)


texts = (
    df["content"]
    .fillna("")
    .astype(str)
)

# Opcional: filtra vacíos muy cortos (ruido)
mask = texts.str.len() >= 10
df = df[mask].copy()
texts = texts[mask].tolist()

len(df), df["content"].head()

(4399,
 0    ['', 'Estoy leyendo el último\xa0 libro que ha...
 1    ['', 'Hola,para aquellos que vivan en BCN o al...
 2    ['', 'Llevaba 10 años comprando tiras en Amazo...
 3    ['', 'Yo creo q mi hija compra alguna en otras...
 4    ['', 'Me salen estas:LinkPero si,\xa0 hay poca...
 Name: content, dtype: object)

In [2]:
from sentence_transformers import SentenceTransformer

model_name = "paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(model_name)

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings.shape

C:\Users\carme\anaconda3\envs\tfg\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 69/69 [01:20<00:00,  1.17s/it]


(4399, 384)

In [3]:
import numpy as np
from pathlib import Path

Path("models").mkdir(parents=True, exist_ok=True)

np.save(Config.embeddings2_path.value, embeddings)
df.to_csv("models/posts_filtered.csv", index=False, encoding="utf-8")

In [4]:
import numpy as np
embeddings = np.load(Config.embeddings2_path.value)

In [5]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def semantic_search(query, top_k=10):
    q_emb = model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    sims = cosine_similarity(q_emb, embeddings)[0]
    top_idx = np.argsort(-sims)[:top_k]
    return df.iloc[top_idx][["content"]].assign(score=sims[top_idx])

semantic_search("no me funciona el sensor tras 5 dias", top_k=5)

,content,score
160,"['', 'Buenas a todosEn mayo del 2021 comencé c...",0.738197
105,"['', '@AnaisabelNo se si leiste mi último hilo...",0.697818
1000,"['', 'Es ""frecuente"" que esté a 53 o 54 el pri...",0.686388
1685,"['', 'Mi caso es que el sábado dia 28 gestioné...",0.676298
3371,"['', 'El problema de los sensores que no pegan...",0.673977
